# G03: Understanding Attention Heads

**Prerequisites:** G02 (loading GPT-2, running with cache).

**What you'll learn:**
- How to visualize and interpret attention patterns
- Head types: previous-token, induction, and copying heads
- QK circuits: what does a head attend to?
- OV circuits: what does a head write?
- How to measure head importance with direct logit attribution
- Induction heads as a two-layer circuit

Every attention head in GPT-2 has a specific job. This tutorial teaches you how to figure out what that job is.

---

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.circuits import ov_circuit, qk_circuit, full_ov_circuit, direct_logit_attribution
from irtk.head_analysis import find_induction_heads, find_previous_token_heads, classify_heads
from irtk.attention_utils import entropy, all_head_entropy

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"GPT-2 Small: {model.cfg.n_layers} layers x {model.cfg.n_heads} heads = {model.cfg.n_layers * model.cfg.n_heads} total heads")

## What Attention Heads Do

Each attention head reads from the residual stream at every token position. It computes **attention weights** — a distribution over positions — to decide WHERE to look, then copies information from the attended positions and writes it back to the residual stream.

This splits neatly into two circuits. The **QK circuit** (W_Q^T @ W_K) determines the attention pattern: which positions attend to which. The **OV circuit** (W_V @ W_O) determines what information flows through the head once the attention pattern is fixed. Understanding what a head *does* means understanding both circuits.

In [ ]:
# Run model with cache on a sentence with repeated names (good for seeing head diversity)
text = "When Mary and John went to the store, John gave a drink to"
tokens = model.to_tokens(text)
token_labels = [tokenizer.decode([t]) for t in np.array(tokens)]

logits, cache = model.run_with_cache(tokens)
print(f"Input: {text!r}")
print(f"Tokens ({len(tokens)}): {token_labels}")
print(f"Prediction: {tokenizer.decode([int(jnp.argmax(logits[-1]))])}")

# Show attention patterns for 4 heads with visibly different behaviors
# We'll pick heads from different layers to show variety
heads_to_show = [(0, 1), (1, 10), (5, 5), (9, 9)]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, (layer, head) in zip(axes.flat, heads_to_show):
    pattern = np.array(cache[("pattern", layer)][head])  # [q, k]
    im = ax.imshow(pattern, cmap='Blues', vmin=0)
    ax.set_title(f"L{layer}H{head}", fontsize=12)
    ax.set_xticks(range(len(token_labels)))
    ax.set_xticklabels(token_labels, rotation=90, fontsize=7)
    ax.set_yticks(range(len(token_labels)))
    ax.set_yticklabels(token_labels, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Attention Patterns — Four Heads with Different Behaviors", fontsize=14)
plt.tight_layout()
plt.show()

## Reading Attention Patterns

Each heatmap is an attention matrix where **row = query position** (where information flows TO) and **column = key position** (where information flows FROM). A bright cell at row *i*, column *j* means position *i* attends strongly to position *j*.

Look for characteristic shapes: a shifted diagonal (one below the main diagonal) means the head attends to the previous token. A vertical stripe means many positions attend to the same token. A broad wash means diffuse attention spread across many positions.

In [ ]:
# Attention entropy for all heads
# Low entropy = focused attention (sharp, attending to few positions)
# High entropy = diffuse attention (spread across many positions)
ent = all_head_entropy(cache, model)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(ent, aspect='auto', cmap='viridis')
ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_xticks(range(model.cfg.n_heads))
ax.set_yticks(range(model.cfg.n_layers))
ax.set_title("Mean Attention Entropy by Head\n(Dark = focused, Bright = diffuse)")
plt.colorbar(im, ax=ax, label="Entropy (nats)")
plt.tight_layout()
plt.show()

# Find extremes
min_idx = np.unravel_index(np.argmin(ent), ent.shape)
max_idx = np.unravel_index(np.argmax(ent), ent.shape)
print(f"Most focused head: L{min_idx[0]}H{min_idx[1]} (entropy = {ent[min_idx]:.3f})")
print(f"Most diffuse head: L{max_idx[0]}H{max_idx[1]} (entropy = {ent[max_idx]:.3f})")

## Head Types

Not all heads are created equal. Research has identified several classic head types:

1. **Previous-token heads**: Attend to the position immediately before the query (shifted diagonal pattern). These serve as a basic information-routing mechanism — "tell me what came right before."

2. **Induction heads**: Implement in-context copying. When the model sees a pattern like `...A B ... A`, an induction head attends from the second `A` to the token after the first `A` (which is `B`), predicting that `B` will follow again.

3. **Copying / name mover heads**: Attend to a specific token (like a name) and copy its identity to the output, promoting that token in the logits.

In [ ]:
# Find previous-token heads
prev_heads = find_previous_token_heads(cache, model, threshold=0.3)

print("Previous-token heads (score = avg attention to position i-1):")
for layer, head, score in prev_heads[:10]:
    print(f"  L{layer}H{head}: score = {score:.3f}")

# Visualize the top previous-token head
if prev_heads:
    best_l, best_h, best_score = prev_heads[0]
    pattern = np.array(cache[("pattern", best_l)][best_h])

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(pattern, cmap='Blues', vmin=0)
    ax.set_title(f"Top Previous-Token Head: L{best_l}H{best_h} (score={best_score:.3f})\nNote the shifted diagonal")
    ax.set_xticks(range(len(token_labels)))
    ax.set_xticklabels(token_labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(token_labels)))
    ax.set_yticklabels(token_labels, fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Find induction heads using repeated random tokens
induction_heads = find_induction_heads(model, seq_len=50, threshold=0.3)

print("Induction heads (score = avg attention to induction-offset position):")
for layer, head, score in induction_heads[:10]:
    print(f"  L{layer}H{head}: score = {score:.3f}")

# Now show the top induction head on a sequence with repeated words
if induction_heads:
    best_l, best_h, best_score = induction_heads[0]

    # Create a repeated sequence to make induction visible
    repeat_text = "The cat sat on the mat and the cat sat on the mat and"
    repeat_tokens = model.to_tokens(repeat_text)
    repeat_labels = [tokenizer.decode([t]) for t in np.array(repeat_tokens)]
    _, repeat_cache = model.run_with_cache(repeat_tokens)

    pattern = np.array(repeat_cache[("pattern", best_l)][best_h])

    fig, ax = plt.subplots(figsize=(10, 9))
    ax.imshow(pattern, cmap='Blues', vmin=0)
    ax.set_title(f"Top Induction Head: L{best_l}H{best_h} (score={best_score:.3f})\nRepeated sequence — notice attending to tokens after previous occurrences")
    ax.set_xticks(range(len(repeat_labels)))
    ax.set_xticklabels(repeat_labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(repeat_labels)))
    ax.set_yticklabels(repeat_labels, fontsize=8)
    plt.tight_layout()
    plt.show()

## The Induction Circuit

Induction heads don't work alone — they're the second half of a two-layer circuit:

1. **Layer L** has a **previous-token head** that writes "the token before me was X" into the residual stream at each position.
2. **Layer L+1** has an **induction head** whose QK circuit reads that signal and attends to positions where "the token before was the same as my current token."
3. Together they implement the rule: *if you've seen this token before, predict what came after it last time.* This is the simplest known circuit for in-context learning.

This is a beautiful example of **composition** — one head's output becomes another head's input, creating a capability neither could achieve alone.

In [ ]:
# Classify all heads by behavior
categories = classify_heads(model, tokens)

print("Head Classification Results")
print("=" * 50)
for category, heads in categories.items():
    print(f"\n{category} ({len(heads)} heads):")
    if heads:
        head_strs = [f"L{l}H{h}" for l, h in heads]
        # Print in rows of 10 for readability
        for i in range(0, len(head_strs), 10):
            print(f"  {', '.join(head_strs[i:i+10])}")

## QK Circuits — What Does the Head Attend To?

The QK circuit is W_Q^T @ W_K — it determines the attention pattern before softmax. By computing the SVD (singular value decomposition) of this matrix, we can see the dominant directions that drive attention. A small number of large singular values means the head's attention is controlled by a few key features. A flat spectrum means it uses many features roughly equally.

In [ ]:
# Pick an interesting head — use the top previous-token head if available,
# otherwise fall back to a known interesting head
if prev_heads:
    qk_layer, qk_head = prev_heads[0][0], prev_heads[0][1]
else:
    qk_layer, qk_head = 0, 1

qk = qk_circuit(model, qk_layer, qk_head)
print(f"QK circuit for L{qk_layer}H{qk_head}: {qk}")

# Singular values reveal the effective rank of the attention computation
_, S, _ = qk.svd()
S = np.array(S)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(S, 'o-', markersize=3)
ax1.set_xlabel("Singular Value Index")
ax1.set_ylabel("Singular Value")
ax1.set_title(f"QK Singular Values — L{qk_layer}H{qk_head}")
ax1.grid(True, alpha=0.3)

# Cumulative explained variance
cumvar = np.cumsum(S**2) / np.sum(S**2)
ax2.plot(cumvar, 'o-', markersize=3)
ax2.axhline(y=0.9, color='r', linestyle='--', alpha=0.5, label='90% variance')
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance Explained")
ax2.set_title(f"QK Effective Rank — L{qk_layer}H{qk_head}")
ax2.legend()
ax2.grid(True, alpha=0.3)

n_90 = int(np.searchsorted(cumvar, 0.9)) + 1
print(f"\n{n_90} components explain 90% of QK variance (out of {len(S)} total)")
print(f"Top 5 singular values: {S[:5]}")

plt.tight_layout()
plt.show()

## OV Circuits — What Does the Head Copy?

The OV circuit is W_V @ W_O — it determines what information flows through the head once the attention pattern is decided. The **full OV circuit** (W_E @ W_V @ W_O @ W_U) maps from input vocabulary to output vocabulary: element [i, j] tells you "if this head attends to token *i*, how much does that promote token *j* in the output logits?"

This is incredibly powerful for interpretation. A head that promotes the same token it attends to is a **copying head**. A head that promotes related tokens is doing something more subtle.

In [ ]:
# Analyze the full OV circuit: what does attending to a token promote in the output?
# Pick a head from a later layer (these tend to have clearer token-level effects)
ov_layer, ov_head = 9, 9  # A later-layer head

fov = full_ov_circuit(model, ov_layer, ov_head)
print(f"Full OV circuit for L{ov_layer}H{ov_head}: {fov}")

# Materialize the full matrix and look at specific source tokens
ov_matrix = np.array(fov.AB)  # [d_vocab, d_vocab]

# For a few interesting tokens, show what the head promotes when attending to them
probe_words = [" Mary", " John", " the", " Paris", " dog"]
probe_ids = [tokenizer.encode(w)[0] for w in probe_words]

print(f"\nL{ov_layer}H{ov_head} OV circuit: 'If I attend to token X, I promote...'")
print("=" * 70)
for word, tok_id in zip(probe_words, probe_ids):
    row = ov_matrix[tok_id]  # effect of attending to this token
    top_k = np.argsort(row)[::-1][:5]
    bot_k = np.argsort(row)[:5]
    
    promoted = [f"{tokenizer.decode([t])!r}({row[t]:+.2f})" for t in top_k]
    suppressed = [f"{tokenizer.decode([t])!r}({row[t]:+.2f})" for t in bot_k]
    
    print(f"\n  Attend to {word!r} (token {tok_id}):")
    print(f"    Promotes:   {', '.join(promoted)}")
    print(f"    Suppresses: {', '.join(suppressed)}")

## Direct Logit Attribution

Direct logit attribution (DLA) measures each head's direct contribution to the logit of a specific target token. Each head writes a vector to the residual stream via `z @ W_O`. We project this through the unembedding to get the head's effect on the logits.

This tells us: for a given prediction, which heads are pushing the model *toward* the answer (positive DLA) and which are pushing *away* (negative DLA)?

In [ ]:
# Direct logit attribution: which heads contribute to predicting " Paris"?
text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)
token_labels = [tokenizer.decode([t]) for t in np.array(tokens)]

logits, cache = model.run_with_cache(tokens)
pred = int(jnp.argmax(logits[-1]))
print(f"Input: {text!r}")
print(f"Model predicts: {tokenizer.decode([pred])!r}")

# Measure each head's contribution to the predicted token's logit
target_token = tokenizer.encode(" Paris")[0]
dla = direct_logit_attribution(model, cache, token=target_token, pos=-1)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(dla, aspect='auto', cmap='RdBu_r')
ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_xticks(range(model.cfg.n_heads))
ax.set_yticks(range(model.cfg.n_layers))
ax.set_title(f"Direct Logit Attribution for ' Paris'\n(Red = promotes, Blue = suppresses)")
plt.colorbar(im, ax=ax, label="Logit Contribution")

# Annotate the strongest heads
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        if abs(dla[layer, head]) > 0.5:
            ax.text(head, layer, f"{dla[layer, head]:.1f}",
                   ha='center', va='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.show()

# Top contributing heads
print("\nTop 10 heads promoting ' Paris':")
flat = dla.flatten()
for i in np.argsort(flat)[::-1][:10]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: {dla[l, h]:+.3f}")

print("\nTop 5 heads suppressing ' Paris':")
for i in np.argsort(flat)[:5]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: {dla[l, h]:+.3f}")

## Key Takeaways

1. **Attention patterns show WHERE information flows** — heatmaps reveal what each head looks at, and characteristic shapes map to head types.

2. **QK circuits determine what features drive attention** — SVD of the QK matrix reveals the low-rank structure behind attention decisions.

3. **OV circuits determine what information gets copied** — the full OV circuit (embedding → value → output → unembedding) shows the vocabulary-level effect of each head.

4. **Head types correspond to specific circuit roles** — previous-token heads route positional information, induction heads enable in-context copying, and name movers copy specific tokens.

5. **Direct logit attribution identifies the most responsible heads** — DLA projects each head's output onto the target logit direction, revealing which heads drive a prediction.

## Exercises

1. **Entropy extremes:** Find the head with the highest entropy and the lowest — visualize their attention patterns side by side. What do they look like?

2. **Factual attribution:** Pick a factual prompt (e.g., "The capital of Germany is") and find which heads contribute most to the correct answer using `direct_logit_attribution`.

3. **Negative DLA:** Do any heads have strongly *negative* direct logit attribution for the correct token? What might they be doing? (Hint: think about competition between candidate answers.)

4. **Classification stability:** Run `classify_heads` on several different prompts. Do the classifications change? Which head types are most stable?

## Further Reading

- Elhage et al. 2021, ["A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — the foundational paper on QK/OV circuit decomposition.
- Olsson et al. 2022, ["In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — the definitive study of induction heads and their role in in-context learning.
- **Next:** G04 — Finding Circuits: The IOI Case Study.